# Digit Recognition Using Support Vector Machine (SVM)
### Kaggle MNIST Dataset – Complete ML Pipeline

### 1. Importing Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

: 

### 2. Load and Explore the Dataset

In [ ]:
## Load dataset
df = pd.read_csv("train.csv")
df.head()

In [ ]:
## Shape of dataset
df.shape

In [ ]:
## Dataset info
df.info()

In [ ]:
## Check for null values
df.isnull().sum().sum()

In [ ]:
## Class distribution
df["label"].value_counts().sort_index()

### 3. Visualize Sample Digit Images

In [ ]:
## Plot sample images from the dataset
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Sample Digit Images from MNIST", fontsize=14)

for i, ax in enumerate(axes.flatten()):
    img = df.iloc[i, 1:].values.reshape(28, 28)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Label: {df.iloc[i, 0]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 4. Data Preprocessing

In [ ]:
### Independent and Dependent features
X = df.drop("label", axis=1)
y = df["label"]

In [ ]:
X.shape, y.shape

In [ ]:
## Normalize pixel values between 0 and 1
X = X / 255.0
X.head()

In [ ]:
## Split the dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                     test_size=0.20,
                                                     random_state=42,
                                                     stratify=y)
print("Training samples :", X_train.shape[0])
print("Testing  samples :", X_test.shape[0])

### 5. Model Building – SVM (Linear & RBF Kernel)
> **Note :** Training is done on 10 000 samples for the kernel comparison (SVM scales as O(n²–n³)). The optimised model uses the full set.

In [ ]:
## Subset for fast kernel comparison
import numpy as np
np.random.seed(42)
idx = np.random.choice(X_train.shape[0], 10000, replace=False)
X_tr_small = X_train.iloc[idx]
y_tr_small = y_train.iloc[idx]

#### 5a. Linear Kernel

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

classifier_linear = SVC(kernel="linear", C=1.0, random_state=42)
classifier_linear.fit(X_tr_small, y_tr_small)
y_pred_linear = classifier_linear.predict(X_test)
accuracy_score(y_test, y_pred_linear)

#### 5b. RBF Kernel

In [ ]:
classifier_rbf = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=42)
classifier_rbf.fit(X_tr_small, y_tr_small)
y_pred_rbf = classifier_rbf.predict(X_test)
accuracy_score(y_test, y_pred_rbf)

### 6. Model Evaluation

In [ ]:
## Classification report – Linear kernel
print("Classification Report (Linear Kernel)")
print(classification_report(y_test, y_pred_linear))

In [ ]:
## Classification report – RBF kernel
print("Classification Report (RBF Kernel)")
print(classification_report(y_test, y_pred_rbf))

In [ ]:
## Confusion matrix – RBF kernel
cm = confusion_matrix(y_test, y_pred_rbf)

plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=range(10), yticklabels=range(10))
plt.title("Confusion Matrix – SVM (RBF Kernel)", fontsize=13)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

### 7. Hyperparameter Tuning – GridSearchCV

In [ ]:
## Subset for GridSearch (keeps runtime reasonable)
idx_g = np.random.choice(X_tr_small.shape[0], 5000, replace=False)
X_g = X_tr_small.iloc[idx_g]
y_g = y_tr_small.iloc[idx_g]

In [ ]:
param_grid = {
    "C"      : [1, 10, 50],
    "gamma"  : ["scale", "auto"],
    "kernel" : ["rbf", "linear"]
}

grid_search = GridSearchCV(SVC(random_state=42),
                           param_grid,
                           cv=3,
                           scoring="accuracy",
                           n_jobs=-1,
                           verbose=1)
grid_search.fit(X_g, y_g)

In [ ]:
## Best parameters and best CV score
print("Best Parameters :", grid_search.best_params_)
print("Best CV Accuracy:", round(grid_search.best_score_ * 100, 2), "%")

In [ ]:
## Retrain best model on full training set
classifier = SVC(**grid_search.best_params_, random_state=42)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
print("Test Accuracy (optimised):", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

In [ ]:
## Full classification report – optimised model
print(classification_report(y_test, y_pred))

### 8. Visualisation – Sample Predictions

In [ ]:
## Show correct and wrong predictions
y_pred_arr  = np.array(y_pred)
y_test_arr  = np.array(y_test)

correct = np.where(y_pred_arr == y_test_arr)[0]
wrong   = np.where(y_pred_arr != y_test_arr)[0]

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
fig.suptitle("Sample Predictions   |   Green = Correct   Red = Wrong", fontsize=13)

for i, ax in enumerate(axes.flatten()):
    if i < 10:
        idx  = correct[i]
        col  = "green"
    else:
        idx  = wrong[i - 10] if len(wrong) > (i - 10) else correct[i]
        col  = "red"

    img = X_test.iloc[idx].values.reshape(28, 28)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"T:{y_test_arr[idx]}  P:{y_pred_arr[idx]}", color=col, fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor(col)
        spine.set_linewidth(2)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
## Confusion matrix – optimised model
cm_opt = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(11, 8))
sns.heatmap(cm_opt, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=range(10), yticklabels=range(10),
            linewidths=0.5)
plt.title("Confusion Matrix – Optimised SVM", fontsize=13)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

### 9. PCA – 2D Visualisation of Digit Classes

In [ ]:
## Apply PCA to reduce 784 dimensions → 2
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train.sample(5000, random_state=42))
y_pca = y_train.iloc[:5000].values

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=y_pca, cmap="tab10", alpha=0.5, s=10)
plt.colorbar(scatter, label="Digit Class")
plt.title("PCA 2D Projection of MNIST Digits", fontsize=13)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

In [ ]:
## Variance explained by 2 components
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance Explained :", round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

### 10. Optional – Compare SVM with Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

## Train Logistic Regression
lr = LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", round(acc_lr * 100, 2), "%")

In [ ]:
## Bar chart – model comparison
models  = ["SVM (linear)", "SVM (rbf)", "SVM (optimised)", "Logistic Regression"]
scores  = [
    accuracy_score(y_test, y_pred_linear) * 100,
    accuracy_score(y_test, y_pred_rbf)    * 100,
    accuracy_score(y_test, y_pred)        * 100,
    acc_lr * 100
]
colours = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

plt.figure(figsize=(9, 5))
bars = plt.bar(models, scores, color=colours, edgecolor="black", linewidth=0.8)

for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f"{score:.2f}%", ha="center", fontsize=11)

plt.ylim(0, 105)
plt.ylabel("Accuracy (%)", fontsize=11)
plt.title("Model Comparison: SVM vs Logistic Regression", fontsize=13)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### 11. Conclusion

| Model | Accuracy |
|---|---|
| SVM (linear kernel) | ~92–94 % |
| SVM (RBF kernel) | ~96–97 % |
| SVM (optimised – GridSearch) | ~97–98 % |
| Logistic Regression | ~91–93 % |

**Strengths of SVM for Digit Recognition**
- The kernel trick (RBF) handles non-linear class boundaries extremely well
- Robust to overfitting when  is tuned properly
- Competitive accuracy even without deep learning

**Limitations of SVM**
- Training time scales as O(n²–n³) – slow on 42 k+ samples
- Prediction latency grows with the number of support vectors
- No spatial invariance; CNNs still outperform SVMs on raw pixels (~99 %+)
- Hyperparameter search (C, γ) adds significant computation cost

> For production digit recognition, a CNN (e.g. LeNet-5) is preferred.  
> SVM remains an excellent, interpretable baseline for smaller datasets.


---
### 12. 🖥️  GUI – Draw & Predict (Tkinter)
> Run the cells below **after** training is complete. They save the model and launch an interactive digit-recognizer window.

In [ ]:
# ── Auto-fix tkinter on macOS Homebrew Python ─────────────────────────
import subprocess, sys

def ensure_tkinter():
    try:
        import tkinter
        print(f'✅  tkinter OK  (Python {sys.version_info.major}.{sys.version_info.minor})')
    except ModuleNotFoundError:
        py_ver = f'{sys.version_info.major}.{sys.version_info.minor}'
        pkg    = f'python-tk@{py_ver}'
        print(f'⚠️  tkinter missing → installing {pkg} via Homebrew …')
        result = subprocess.run(
            ['brew', 'install', pkg],
            capture_output=True, text=True
        )
        output = result.stdout + result.stderr
        print(output)
        if result.returncode == 0:
            print('✅  Installed!  Please RESTART the kernel, then re-run from this cell.')
        else:
            print('❌  brew install failed. Try manually:')
            print(f'       brew install {pkg}')
        sys.exit(0)

ensure_tkinter()


In [ ]:
# ── Save the trained SVM model to disk ──────────────────────────────────
import joblib, os

MODEL_PATH = 'svm_digit_model.pkl'

# 'classifier' = the optimised SVM retrained in Section 7.
# Change the variable name below if yours is different.
joblib.dump(classifier, MODEL_PATH)
print(f'✅  Model saved → {os.path.abspath(MODEL_PATH)}')
print(f'    Model type : {type(classifier).__name__}')
print(f'    File size  : {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  SVM Digit Recognizer – complete Tkinter GUI (inline in notebook)
# ════════════════════════════════════════════════════════════════════

import tkinter as tk
from tkinter import font as tkfont
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageOps
import joblib, os, sys, threading

# ── Constants ────────────────────────────────────────────────────────
MODEL_PATH   = 'svm_digit_model.pkl'
CANVAS_SIZE  = 400       # drawing area (px)
BRUSH_RADIUS = 14        # half-width of brush stroke
BRUSH_COLOR  = 'white'   # digit colour  (black canvas bg)
BG_COLOR     = 'black'
TARGET_SIZE  = (28, 28)  # MNIST standard


# ── 1. Model loader ──────────────────────────────────────────────────
def load_model(path):
    """Load pre-trained SVM from disk."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Model file '{path}' not found.\n"
            "Run the 'Save model' cell first."
        )
    return joblib.load(path)


# ── 2. Pre-processing pipeline ───────────────────────────────────────
def preprocess(pil_image):
    """
    Raw canvas image (400x400 RGB)
      → grayscale → crop to bounding box → square-pad
      → resize 28x28 → Gaussian blur → normalise ÷255 → flatten (784,)
    """
    gray = pil_image.convert('L')
    arr  = np.array(gray)
    nz   = np.argwhere(arr > 10)

    if nz.size == 0:
        return np.zeros(784, dtype=np.float32)   # blank canvas

    y0, x0 = nz.min(axis=0)
    y1, x1 = nz.max(axis=0)
    pad     = max(int(max(y1 - y0, x1 - x0) * 0.20), 10)
    y0 = max(y0 - pad, 0);  x0 = max(x0 - pad, 0)
    y1 = min(y1 + pad, arr.shape[0]);  x1 = min(x1 + pad, arr.shape[1])
    cropped = gray.crop((x0, y0, x1, y1))

    # Make square
    cw, ch  = cropped.size
    side    = max(cw, ch)
    square  = Image.new('L', (side, side), 0)
    square.paste(cropped, ((side - cw) // 2, (side - ch) // 2))

    resized  = square.resize(TARGET_SIZE, Image.LANCZOS)
    blurred  = resized.filter(ImageFilter.GaussianBlur(radius=0.8))
    return (np.array(blurred, dtype=np.float32) / 255.0).flatten()


# ── 3. Main GUI class ─────────────────────────────────────────────────
class DigitRecognizerApp:

    def __init__(self, root, model):
        self.root  = root
        self.model = model
        self._pil_image = Image.new('RGB', (CANVAS_SIZE, CANVAS_SIZE), BG_COLOR)
        self._pil_draw  = ImageDraw.Draw(self._pil_image)
        self._last_x = self._last_y = None
        self._build_ui()

    # ── UI layout ────────────────────────────────────────────────────
    def _build_ui(self):
        self.root.title('SVM Digit Recognizer')
        self.root.resizable(False, False)
        self.root.configure(bg='#1e1e2e')

        # Title
        tk.Label(self.root, text='✏️  Digit Recognizer',
                 font=tkfont.Font(family='Helvetica', size=18, weight='bold'),
                 fg='#cdd6f4', bg='#1e1e2e').pack(pady=(16, 2))
        tk.Label(self.root, text='Draw a digit (0 – 9) on the canvas',
                 font=tkfont.Font(family='Helvetica', size=10),
                 fg='#a6adc8', bg='#1e1e2e').pack(pady=(0, 10))

        # Canvas
        cf = tk.Frame(self.root, bg='#313244', bd=3, relief='solid')
        cf.pack(padx=30)
        self.canvas = tk.Canvas(cf, width=CANVAS_SIZE, height=CANVAS_SIZE,
                                bg=BG_COLOR, cursor='crosshair', highlightthickness=0)
        self.canvas.pack()
        self.canvas.bind('<ButtonPress-1>',   self._press)
        self.canvas.bind('<B1-Motion>',       self._drag)
        self.canvas.bind('<ButtonRelease-1>', self._release)

        # Result row
        rf = tk.Frame(self.root, bg='#1e1e2e')
        rf.pack(pady=16)
        self.result_lbl = tk.Label(rf, text='—',
                                   font=tkfont.Font(family='Helvetica', size=64, weight='bold'),
                                   fg='#a6e3a1', bg='#1e1e2e', width=3)
        self.result_lbl.grid(row=0, column=0, padx=(0, 20))

        inf = tk.Frame(rf, bg='#1e1e2e')
        inf.grid(row=0, column=1, sticky='w')
        tk.Label(inf, text='Predicted Digit',
                 font=tkfont.Font(family='Helvetica', size=10),
                 fg='#a6adc8', bg='#1e1e2e').pack(anchor='w')
        self.conf_lbl = tk.Label(inf, text='Confidence: —',
                                 font=tkfont.Font(family='Helvetica', size=12, weight='bold'),
                                 fg='#89dceb', bg='#1e1e2e')
        self.conf_lbl.pack(anchor='w', pady=(4, 0))

        # Probability bars
        self._prob_frame = tk.Frame(self.root, bg='#1e1e2e')
        self._prob_frame.pack(pady=(0, 10), padx=30, fill='x')
        self._bars = []
        self._bar_labels = []
        for d in range(10):
            row = tk.Frame(self._prob_frame, bg='#1e1e2e')
            row.pack(fill='x', pady=1)
            tk.Label(row, text=str(d), font=('Courier', 9, 'bold'),
                     fg='#cdd6f4', bg='#1e1e2e', width=2).pack(side='left')
            bc = tk.Canvas(row, width=240, height=10, bg='#313244', highlightthickness=0)
            bc.pack(side='left', padx=(4, 6))
            br = bc.create_rectangle(0, 0, 0, 10, fill='#89b4fa', width=0)
            pl = tk.Label(row, text='', font=('Courier', 8),
                          fg='#a6adc8', bg='#1e1e2e', width=6, anchor='w')
            pl.pack(side='left')
            self._bars.append((bc, br))
            self._bar_labels.append(pl)

        # Buttons
        bf = tk.Frame(self.root, bg='#1e1e2e')
        bf.pack(pady=(0, 20))
        tk.Button(bf, text='  Predict  ',
                  font=tkfont.Font(family='Helvetica', size=13, weight='bold'),
                  fg='#1e1e2e', bg='#a6e3a1', activebackground='#94e2b5',
                  relief='flat', cursor='hand2', padx=16, pady=8,
                  command=self.predict).grid(row=0, column=0, padx=10)
        tk.Button(bf, text='  Clear  ',
                  font=tkfont.Font(family='Helvetica', size=13, weight='bold'),
                  fg='#1e1e2e', bg='#f38ba8', activebackground='#eba0ac',
                  relief='flat', cursor='hand2', padx=16, pady=8,
                  command=self.clear).grid(row=0, column=1, padx=10)

        # Status bar
        self.status = tk.StringVar(value='Ready  –  Draw a digit and press Predict')
        tk.Label(self.root, textvariable=self.status,
                 font=tkfont.Font(family='Helvetica', size=9),
                 fg='#585b70', bg='#181825', anchor='w', padx=10, pady=4
                 ).pack(fill='x', side='bottom')

    # ── Drawing helpers ──────────────────────────────────────────────
    def _circle(self, x, y):
        r = BRUSH_RADIUS
        self.canvas.create_oval(x-r, y-r, x+r, y+r, fill=BRUSH_COLOR, outline='')
        self._pil_draw.ellipse([x-r, y-r, x+r, y+r], fill=BRUSH_COLOR)

    def _press(self, e):
        self._last_x, self._last_y = e.x, e.y
        self._circle(e.x, e.y)

    def _drag(self, e):
        if self._last_x is None:
            return
        self.canvas.create_line(self._last_x, self._last_y, e.x, e.y,
                                fill=BRUSH_COLOR, width=BRUSH_RADIUS*2,
                                capstyle=tk.ROUND, joinstyle=tk.ROUND, smooth=True)
        self._pil_draw.line([self._last_x, self._last_y, e.x, e.y],
                            fill=BRUSH_COLOR, width=BRUSH_RADIUS*2)
        self._circle(e.x, e.y)
        self._circle(self._last_x, self._last_y)
        self._last_x, self._last_y = e.x, e.y

    def _release(self, e):
        self._last_x = self._last_y = None

    # ── Probability bars update ──────────────────────────────────────
    def _update_bars(self, proba, pred):
        for i, ((bc, br), lbl) in enumerate(zip(self._bars, self._bar_labels)):
            if proba is None:
                bc.coords(br, 0, 0, 0, 10); lbl.config(text=''); continue
            pct = float(proba[i]) * 100
            bc.coords(br, 0, 0, int(pct / 100 * 240), 10)
            bc.itemconfig(br, fill='#a6e3a1' if i == pred else '#89b4fa')
            lbl.config(text=f'{pct:5.1f}%')

    # ── Predict ──────────────────────────────────────────────────────
    def predict(self):
        self.status.set('⏳  Processing …')
        self.root.update_idletasks()

        feat = preprocess(self._pil_image)
        if feat.sum() == 0:
            self.status.set('⚠️  Canvas is empty – draw a digit first')
            return

        x    = feat.reshape(1, -1)
        pred = int(self.model.predict(x)[0])
        proba, conf_text = None, ''

        if hasattr(self.model, 'predict_proba'):
            proba     = self.model.predict_proba(x)[0]
            conf_text = f'Confidence: {proba[pred]*100:.1f}%'
        elif hasattr(self.model, 'decision_function'):
            scores = self.model.decision_function(x)[0]
            e      = np.exp(scores - scores.max())
            proba  = e / e.sum()
            conf_text = f'Confidence: {proba[pred]*100:.1f}%  (approx.)'

        self.result_lbl.config(text=str(pred))
        self.conf_lbl.config(text=conf_text or 'Confidence: N/A')
        self._update_bars(proba, pred)
        self.status.set(f'✅  Predicted digit: {pred}')

    # ── Clear ─────────────────────────────────────────────────────────
    def clear(self):
        self.canvas.delete('all')
        self._pil_image = Image.new('RGB', (CANVAS_SIZE, CANVAS_SIZE), BG_COLOR)
        self._pil_draw  = ImageDraw.Draw(self._pil_image)
        self.result_lbl.config(text='—')
        self.conf_lbl.config(text='Confidence: —')
        self._update_bars(None, None)
        self.status.set('🗑️  Cleared – draw a digit and press Predict')


# ── 4. Launch ─────────────────────────────────────────────────────────
def launch_gui():
    model = load_model(MODEL_PATH)
    root  = tk.Tk()
    DigitRecognizerApp(root, model)
    root.mainloop()


# Open the GUI in a background thread so Jupyter stays responsive
print('🚀  Launching GUI window …')
t = threading.Thread(target=launch_gui, daemon=True)
t.start()
print('✅  GUI launched!  (close the window to stop)')
print('💡  Tip: if the window does not appear, run   python digit_recognizer_gui.py   in a terminal.')
